# Notebook-first application walkthrough

**Problem / objective:** Evaluate product experiments with power, variance reduction, guardrails and uncertainty rather than declaring winners from a p-value alone.

**Decision / solution:** Ship, iterate or stop based on effect size, confidence interval, guardrails and practical business value.

This front section is intentionally analysis-first. It uses direct notebook code for inspection, EDA, visualisation and evidence review. The original notebook work is preserved below, followed by modular production code where that adds engineering evidence.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
PROJECT_SLUG = 'experiment_lab'
ROOT = Path.cwd()
if not (ROOT / 'projects').exists():
    candidate = ROOT.parent.parent if ROOT.name == PROJECT_SLUG else ROOT
    if (candidate / 'projects').exists():
        ROOT = candidate
PROJECT = ROOT / 'projects' / PROJECT_SLUG
if not PROJECT.exists() and Path.cwd().name == PROJECT_SLUG:
    PROJECT = Path.cwd()
    ROOT = PROJECT.parent.parent
assert PROJECT.exists(), f'Project directory not found: {PROJECT}'
print('Repository root:', ROOT.resolve())
print('Project:', PROJECT.resolve())


## 1. Find the real data and retained evidence

Instead of hiding the dataset behind a helper function, start by seeing what the project actually ships: raw/small data, fixtures, outputs, results and verified evidence. External large datasets remain reproducibly downloadable from the documented source.


In [ ]:
candidate_files = []
for pattern in ('*.csv', '*.parquet', '*.json', '*.tsv', '*.txt'):
    candidate_files.extend(PROJECT.rglob(pattern))
verified_dir = ROOT / 'verified' / PROJECT_SLUG
if verified_dir.exists():
    for pattern in ('*.csv', '*.parquet', '*.json', '*.tsv', '*.txt'):
        candidate_files.extend(verified_dir.rglob(pattern))
candidate_files = sorted({p.resolve() for p in candidate_files if p.is_file()})
file_inventory = pd.DataFrame({
    'file': [str(p.relative_to(ROOT)) if ROOT in p.parents else str(p) for p in candidate_files],
    'suffix': [p.suffix.lower() for p in candidate_files],
    'size_kb': [round(p.stat().st_size / 1024, 1) for p in candidate_files],
})
display(file_inventory.head(40))
print(f'Inspectable local data/evidence files: {len(file_inventory):,}')


## 2. Direct tabular data audit

The code below deliberately avoids a project-specific wrapper. It opens the first sensible local tabular asset, shows its schema and quality profile, and makes the data issues visible before modelling. If the full raw dataset is external, run the project's documented download cell/entry point first and rerun this section.


In [ ]:
tabular_candidates = [p for p in candidate_files if p.suffix.lower() in {'.csv', '.tsv', '.parquet'}]
preferred = [p for p in tabular_candidates if not any(token in p.name.lower() for token in ('metric', 'summary', 'verification'))]
tabular_path = (preferred or tabular_candidates or [None])[0]
df = None
if tabular_path is not None:
    if tabular_path.suffix.lower() == '.parquet':
        df = pd.read_parquet(tabular_path)
    else:
        sep = '\t' if tabular_path.suffix.lower() == '.tsv' else ','
        df = pd.read_csv(tabular_path, sep=sep, nrows=200_000)
    print('Loaded:', tabular_path)
    print('Shape:', df.shape)
    display(df.head())
    audit = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'missing': df.isna().sum(),
        'missing_pct': (100 * df.isna().mean()).round(2),
        'unique': df.nunique(dropna=False),
    }).sort_values(['missing_pct', 'unique'], ascending=[False, False])
    display(audit.head(30))
    print('Duplicate rows:', int(df.duplicated().sum()))
else:
    print('No local CSV/TSV/Parquet found yet. Use the project README/run path to download or build the documented dataset, then rerun this audit.')


## 3. Exploratory data analysis and visualisation

These plots are intentionally created in the notebook rather than described in prose. They expose distribution, missingness, scale, category balance and numeric relationships before any final model decision.


In [ ]:
if df is not None and len(df):
    missing_pct = (100 * df.isna().mean()).sort_values(ascending=False).head(20)
    missing_pct = missing_pct[missing_pct > 0]
    if len(missing_pct):
        plt.figure(figsize=(10, 4))
        missing_pct.plot(kind='bar')
        plt.title('Missing values by feature (%)')
        plt.ylabel('Missing %')
        plt.xticks(rotation=60, ha='right')
        plt.tight_layout()
        plt.show()

    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()[:8]
    for col in numeric_cols:
        series = pd.to_numeric(df[col], errors='coerce').dropna()
        if len(series):
            plt.figure(figsize=(8, 4))
            plt.hist(series, bins=30, alpha=0.8)
            plt.axvline(series.median(), linestyle='--', label=f'median={series.median():.2f}')
            plt.title(f'Distribution: {col}')
            plt.xlabel(col)
            plt.ylabel('Count')
            plt.legend()
            plt.tight_layout()
            plt.show()

    categorical_cols = [c for c in df.columns if c not in numeric_cols and df[c].nunique(dropna=False) <= 30][:4]
    for col in categorical_cols:
        counts = df[col].fillna('<missing>').astype(str).value_counts().head(15)
        plt.figure(figsize=(9, 4))
        counts.sort_values().plot(kind='barh')
        plt.title(f'Top categories: {col}')
        plt.xlabel('Rows')
        plt.tight_layout()
        plt.show()

    if len(numeric_cols) >= 2:
        corr = df[numeric_cols].corr(numeric_only=True)
        plt.figure(figsize=(8, 6))
        image = plt.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm')
        plt.colorbar(image, label='Correlation')
        plt.xticks(range(len(corr.columns)), corr.columns, rotation=60, ha='right')
        plt.yticks(range(len(corr.index)), corr.index)
        plt.title('Numeric correlation matrix')
        plt.tight_layout()
        plt.show()

    if len(numeric_cols) >= 2:
        x_col, y_col = numeric_cols[0], numeric_cols[-1]
        sample = df[[x_col, y_col]].dropna().sample(min(3000, len(df.dropna(subset=[x_col, y_col]))), random_state=42)
        if len(sample):
            plt.figure(figsize=(7, 5))
            plt.scatter(sample[x_col], sample[y_col], alpha=0.35, s=18)
            plt.xlabel(x_col)
            plt.ylabel(y_col)
            plt.title(f'{y_col} versus {x_col}')
            plt.tight_layout()
            plt.show()
else:
    print('Run the documented data-build/download path, then rerun this section to render raw-data EDA.')


## 4. Inspect the measured results, not just the code

A portfolio project is stronger when it retains evidence. This section reads machine-readable JSON/CSV outputs and turns scalar metrics into a quick visual comparison.


In [ ]:
json_files = [p for p in candidate_files if p.suffix.lower() == '.json']
metric_rows = []
for path in json_files[:30]:
    try:
        payload = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        continue
    stack = [('', payload)]
    while stack:
        prefix, value = stack.pop()
        if isinstance(value, dict):
            for key, child in value.items():
                stack.append((f'{prefix}.{key}' if prefix else str(key), child))
        elif isinstance(value, (int, float)) and not isinstance(value, bool) and np.isfinite(value):
            metric_rows.append({
                'file': str(path.relative_to(ROOT)) if ROOT in path.parents else str(path),
                'metric': prefix,
                'value': float(value),
            })
metrics_df = pd.DataFrame(metric_rows)
if len(metrics_df):
    display(metrics_df.head(40))
    plot_df = metrics_df[np.isfinite(metrics_df['value'])].copy()
    plot_df = plot_df[plot_df['value'].abs() < 1_000_000].head(20)
    if len(plot_df):
        labels = (plot_df['file'].str.split('/').str[-1] + ' :: ' + plot_df['metric']).tolist()
        plt.figure(figsize=(10, max(4, 0.35 * len(plot_df))))
        plt.barh(range(len(plot_df)), plot_df['value'])
        plt.yticks(range(len(plot_df)), labels)
        plt.title('Retained project metrics / evidence')
        plt.tight_layout()
        plt.show()
else:
    print('No scalar JSON evidence found. Run the project and retain metrics/results before treating it as complete.')


## 5. Reproduce the application

The notebook should be understandable without running anything, but a reviewer can reproduce the canonical application below. The switch is off by default so opening the notebook never triggers a long training job unexpectedly.


In [ ]:
RUN_PROJECT = False
entrypoint = PROJECT / 'run.py'
if RUN_PROJECT and entrypoint.exists():
    subprocess.run([sys.executable, str(entrypoint)], cwd=PROJECT, check=True)
elif entrypoint.exists():
    print(f'Reproduce with: cd {PROJECT} && {sys.executable} run.py')
else:
    print('This project uses a different documented entry point; see README.md in the project folder.')


## 6. Decision / solution

Ship, iterate or stop based on effect size, confidence interval, guardrails and practical business value.

The final recommendation should be tied to the measured validation evidence and error analysis below. A model is not the solution by itself; the solution is the decision process built around it.


# ExperimentLab — Full Python Code

**Hiring purpose:** one project, one notebook, with the actual Python implementation visible. The modular files remain in the repository because that is how production code should be organised; this notebook mirrors those files so a recruiter can inspect the full code without hunting.


## Dataset and reproducibility

Deterministic synthetic randomized-experiment data with a known treatment effect.

The project README/data card documents provenance, constraints and the exact reproduction path. Large third-party raw files are not duplicated in Git when licensing or repository size makes that poor engineering practice.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys

PROJECT_SLUG = 'experiment_lab'
ROOT = Path.cwd()
if not (ROOT / 'projects').exists():
    target = Path('/content/uni_projects')
    if not target.exists():
        subprocess.run(['git', 'clone', 'https://github.com/Jorgoluka100/uni_projects.git', str(target)], check=True)
    os.chdir(target)
    ROOT = target
PROJECT = ROOT / 'projects' / PROJECT_SLUG
assert PROJECT.exists(), PROJECT
print('Project:', PROJECT.resolve())


## Full Python implementation

Every code cell below is copied directly from the corresponding `.py` file on the same commit. These cells are intentionally tagged `source-mirror` so the notebook acts as a readable code portfolio while the canonical modules remain testable files.


### `run.py`


In [ ]:
from __future__ import annotations

import argparse
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import norm


def simulate(n: int, seed: int, effect: float) -> pd.DataFrame:
    rng=np.random.default_rng(seed); pre=rng.normal(100,20,n); segment=rng.choice(["new","returning"],n,p=[0.35,0.65]); treatment=rng.integers(0,2,n); noise=rng.normal(0,12,n)
    outcome=25+0.62*pre+4.0*(segment=="returning")+effect*treatment+noise; guardrail=rng.normal(5.0,1.1,n)+0.02*treatment
    return pd.DataFrame({"pre_metric":pre,"segment":segment,"treatment":treatment,"outcome":outcome,"guardrail":guardrail})


def mean_effect(y: np.ndarray,t: np.ndarray)->tuple[float,float,float]:
    yt,yc=y[t==1],y[t==0]; effect=float(yt.mean()-yc.mean()); se=math.sqrt(float(yt.var(ddof=1)/len(yt)+yc.var(ddof=1)/len(yc))); return effect,effect-1.96*se,effect+1.96*se


def cuped_adjust(y:np.ndarray,x:np.ndarray)->tuple[np.ndarray,float]:
    theta=float(np.cov(y,x,ddof=1)[0,1]/np.var(x,ddof=1)); return y-theta*(x-x.mean()),theta


def stratified_bootstrap(y:np.ndarray,t:np.ndarray,rounds:int,seed:int)->tuple[float,float]:
    rng=np.random.default_rng(seed); ti,ci=np.where(t==1)[0],np.where(t==0)[0]; values=[]
    for _ in range(rounds):
        ts=rng.choice(ti,len(ti),replace=True); cs=rng.choice(ci,len(ci),replace=True); values.append(float(y[ts].mean()-y[cs].mean()))
    return tuple(float(x) for x in np.quantile(values,[0.025,0.975]))


def power_and_mde(sd:float,n:int,alpha:float=0.05,power:float=0.80)->dict[str,float]:
    per_arm=n/2; return {"alpha":alpha,"target_power":power,"mde_outcome_units":float((norm.ppf(1-alpha/2)+norm.ppf(power))*sd*math.sqrt(2/per_arm))}


def run(n:int,seed:int,effect:float,output_dir:Path)->dict:
    output_dir.mkdir(parents=True,exist_ok=True); df=simulate(n,seed,effect); y=df.outcome.to_numpy(); t=df.treatment.to_numpy(); pre=df.pre_metric.to_numpy()
    raw=mean_effect(y,t); adjusted,theta=cuped_adjust(y,pre); cuped=mean_effect(adjusted,t); boot=stratified_bootstrap(adjusted,t,1500,seed+99)
    variance_reduction=1-float(np.var(adjusted,ddof=1))/float(np.var(y,ddof=1)); guard=mean_effect(df.guardrail.to_numpy(),t); decision="ship" if cuped[1]>0 and guard[1]>-0.20 else "hold"
    payload={"project":"ExperimentLab","verification_pass":bool(abs(cuped[0]-effect)<1.0 and variance_reduction>0.30),"scope":"deterministic synthetic randomized-experiment methodology demo","rows":n,"known_simulated_effect":effect,"raw_effect":{"estimate":raw[0],"ci95":[raw[1],raw[2]]},"cuped_effect":{"estimate":cuped[0],"ci95":[cuped[1],cuped[2]],"bootstrap_ci95":list(boot),"theta":theta},"variance_reduction":variance_reduction,"guardrail_effect":{"estimate":guard[0],"ci95":[guard[1],guard[2]],"non_inferiority_margin":-0.20},"power":power_and_mde(float(np.std(adjusted,ddof=1)),n),"decision":decision,"rules":["random assignment","fixed-horizon inference","CUPED uses a pre-treatment covariate","bootstrap resamples within treatment arms","guardrail must not cross the pre-declared harm margin"]}
    df.to_csv(output_dir/"experiment_data.csv",index=False); (output_dir/"verification.json").write_text(json.dumps(payload,indent=2),encoding="utf-8"); print(json.dumps(payload,indent=2)); return payload


def self_test()->None:
    out=run(12000,42,2.5,Path("/tmp/experimentlab_selftest")); assert out["verification_pass"]; assert out["variance_reduction"]>0.30; assert out["cuped_effect"]["ci95"][0]<2.5<out["cuped_effect"]["ci95"][1]; print("ExperimentLab self-test passed.")


def main()->int:
    p=argparse.ArgumentParser(); p.add_argument("--rows",type=int,default=20000); p.add_argument("--seed",type=int,default=42); p.add_argument("--effect",type=float,default=2.5); p.add_argument("--output-dir",type=Path,default=Path("experimentlab_artifacts")); p.add_argument("--self-test",action="store_true"); a=p.parse_args()
    if a.self_test: self_test(); return 0
    r=run(a.rows,a.seed,a.effect,a.output_dir); return 0 if r["verification_pass"] else 1

if __name__=="__main__": raise SystemExit(main())


## Run the real project

The cell below executes the canonical project entry point rather than a rewritten toy version. Keep `RUN_PIPELINE = False` when you only want to inspect the notebook; change it to `True` to reproduce the project.


In [ ]:
RUN_PIPELINE = False
if RUN_PIPELINE:
    subprocess.run([sys.executable, 'run.py'], cwd=PROJECT, check=True)
else:
    print(f'Reproduce with: cd {PROJECT} && python run.py')


In [ ]:
evidence = []
for folder in (PROJECT / 'results', ROOT / 'verified' / PROJECT_SLUG):
    if folder.exists():
        evidence.extend(sorted(folder.glob('*.json')))
for path in evidence[:5]:
    print('\n---', path.relative_to(ROOT), '---')
    print(path.read_text(encoding='utf-8')[:12000])


## Interview discussion

Be ready to explain the business problem, dataset provenance, cleaning/preprocessing decisions, leakage controls, modelling or analytical choices, evaluation design, limitations, testing strategy and what you would change in production. The key signal is that the notebook, modular source, tests and retained evidence all tell the same story.


# Deeper exploratory analysis and retained evidence

These direct notebook cells extend the initial EDA with data-quality, scale, relationship, output and error diagnostics. They are intentionally visible here rather than hidden behind project helper functions.


In [ ]:
# Extended data-quality scorecard
if df is not None and len(df):
    quality_rows = []
    for col in df.columns:
        series = df[col]
        row = {
            'feature': col,
            'dtype': str(series.dtype),
            'rows': len(series),
            'missing': int(series.isna().sum()),
            'missing_pct': float(100 * series.isna().mean()),
            'unique': int(series.nunique(dropna=False)),
            'unique_pct': float(100 * series.nunique(dropna=False) / max(len(series), 1)),
        }
        if pd.api.types.is_numeric_dtype(series):
            values = pd.to_numeric(series, errors='coerce').dropna()
            if len(values):
                q1, q3 = values.quantile([0.25, 0.75])
                iqr = q3 - q1
                row.update({
                    'mean': float(values.mean()),
                    'median': float(values.median()),
                    'std': float(values.std()),
                    'p05': float(values.quantile(0.05)),
                    'p95': float(values.quantile(0.95)),
                    'skew': float(values.skew()),
                    'iqr_outliers': int(((values < q1 - 1.5*iqr) | (values > q3 + 1.5*iqr)).sum()),
                })
        quality_rows.append(row)
    deep_quality = pd.DataFrame(quality_rows)
    display(deep_quality.sort_values(['missing_pct','unique'], ascending=[False,False]).head(40))
    if 'iqr_outliers' in deep_quality:
        outlier_view = deep_quality.dropna(subset=['iqr_outliers']).sort_values('iqr_outliers', ascending=False).head(15)
        if len(outlier_view):
            plt.figure(figsize=(10,4))
            plt.bar(outlier_view['feature'], outlier_view['iqr_outliers'])
            plt.title('Potential IQR outliers by feature')
            plt.ylabel('Rows')
            plt.xticks(rotation=60, ha='right')
            plt.tight_layout()
            plt.show()
    card = deep_quality.sort_values('unique', ascending=False).head(20)
    plt.figure(figsize=(10,4))
    plt.bar(card['feature'], card['unique'])
    plt.title('Feature cardinality')
    plt.ylabel('Unique values')
    plt.xticks(rotation=60, ha='right')
    plt.tight_layout()
    plt.show()
    print('Constant columns:', deep_quality.loc[deep_quality['unique'] <= 1, 'feature'].tolist())
    print('High-missing columns:', deep_quality.loc[deep_quality['missing_pct'] >= 30, 'feature'].tolist())
    print('Possible identifier columns:', deep_quality.loc[deep_quality['unique_pct'] >= 95, 'feature'].tolist()[:20])
else:
    print('Materialise the documented dataset to run the extended data-quality scorecard.')


In [ ]:
# Numeric distributions, spread and strongest pairwise relationships
if df is not None and len(df):
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()[:12]
    for col in numeric_cols:
        values = pd.to_numeric(df[col], errors='coerce').dropna()
        if len(values) < 5:
            continue
        clipped = values.clip(values.quantile(0.01), values.quantile(0.99))
        plt.figure(figsize=(8,4))
        plt.hist(clipped, bins=35, alpha=0.82)
        plt.axvline(values.median(), linestyle='--', label=f'median={values.median():.3g}')
        plt.axvline(values.mean(), linestyle=':', label=f'mean={values.mean():.3g}')
        plt.title(f'Distribution: {col} (1st–99th percentile)')
        plt.xlabel(col)
        plt.ylabel('Rows')
        plt.legend()
        plt.tight_layout()
        plt.show()
        plt.figure(figsize=(8,3))
        plt.boxplot(values, vert=False, showfliers=True)
        plt.title(f'Spread / outliers: {col}')
        plt.xlabel(col)
        plt.tight_layout()
        plt.show()
    if len(numeric_cols) >= 2:
        corr = df[numeric_cols].corr(numeric_only=True)
        pairs = []
        for i, left in enumerate(corr.columns):
            for right in corr.columns[i+1:]:
                value = corr.loc[left, right]
                if pd.notna(value):
                    pairs.append({'feature_a': left, 'feature_b': right, 'correlation': float(value), 'abs_correlation': float(abs(value))})
        corr_pairs = pd.DataFrame(pairs).sort_values('abs_correlation', ascending=False) if pairs else pd.DataFrame()
        if len(corr_pairs):
            display(corr_pairs.head(20).round(4))
            for _, pair in corr_pairs.head(4).iterrows():
                sample = df[[pair['feature_a'], pair['feature_b']]].dropna()
                if len(sample) > 3000:
                    sample = sample.sample(3000, random_state=42)
                plt.figure(figsize=(7,5))
                plt.scatter(sample[pair['feature_a']], sample[pair['feature_b']], alpha=0.30, s=16)
                plt.xlabel(pair['feature_a'])
                plt.ylabel(pair['feature_b'])
                plt.title(f"{pair['feature_a']} vs {pair['feature_b']} (r={pair['correlation']:.2f})")
                plt.tight_layout()
                plt.show()
    categorical = [c for c in df.columns if 2 <= df[c].nunique(dropna=False) <= 20][:8]
    for col in categorical:
        counts = df[col].fillna('<missing>').astype(str).value_counts().head(20)
        shares = 100 * counts / counts.sum()
        display(pd.DataFrame({'rows': counts, 'share_pct': shares.round(2)}))
        plt.figure(figsize=(8,4))
        counts.sort_values().plot(kind='barh')
        plt.title(f'Category balance: {col}')
        plt.xlabel('Rows')
        plt.tight_layout()
        plt.show()
else:
    print('Materialise the documented dataset to run distribution diagnostics.')


In [ ]:
# Temporal coverage where date/time fields exist
if df is not None and len(df):
    time_cols = [c for c in df.columns if any(token in str(c).lower() for token in ('date','time','timestamp','datetime'))]
    print('Date/time candidates:', time_cols[:10])
    for col in time_cols[:4]:
        converted = pd.to_datetime(df[col], errors='coerce')
        valid = converted.dropna()
        if len(valid) >= max(10, int(0.25*len(df))):
            print(col, 'range:', valid.min(), '→', valid.max())
            monthly = valid.dt.to_period('M').value_counts().sort_index()
            if len(monthly) > 1:
                plt.figure(figsize=(10,4))
                plt.plot(monthly.index.astype(str), monthly.values, marker='o')
                plt.title(f'Rows over time: {col}')
                plt.ylabel('Rows')
                plt.xticks(rotation=70, ha='right')
                plt.tight_layout()
                plt.show()


## Retained outputs and error analysis

A strong portfolio keeps inspectable evidence. The cells below profile compact result tables and automatically detect prediction-like columns for residual or misclassification analysis.


In [ ]:
# Load compact result/evidence tables
result_tables = []
for base in [PROJECT/'results', PROJECT/'outputs', PROJECT/'artifacts', ROOT/'verified'/PROJECT_SLUG]:
    if not base.exists():
        continue
    for path in sorted(base.rglob('*')):
        if path.is_file() and path.suffix.lower() in {'.csv','.tsv','.parquet'} and path.stat().st_size < 25_000_000:
            try:
                if path.suffix.lower() == '.parquet':
                    table = pd.read_parquet(path)
                else:
                    table = pd.read_csv(path, sep='	' if path.suffix.lower() == '.tsv' else ',')
            except Exception as exc:
                print('Could not read', path.name, '-', exc)
                continue
            result_tables.append((path, table))
            print('
RESULT TABLE:', path.relative_to(ROOT) if ROOT in path.parents else path)
            print('shape=', table.shape)
            display(table.head(15))
            numeric = table.select_dtypes(include=np.number).columns.tolist()[:12]
            if numeric:
                display(table[numeric].describe().T.round(4))
print('Inspectable result tables:', len(result_tables))


In [ ]:
# Automatic regression/classification-style error diagnostics
actual_tokens = ('actual','target','truth','y_true','observed','label')
pred_tokens = ('prediction','predicted','forecast','y_pred')
confidence_tokens = ('confidence','probability','proba','risk','uncertainty')
for path, table in result_tables:
    actual_cols = [c for c in table.columns if any(token in str(c).lower() for token in actual_tokens)]
    pred_cols = [c for c in table.columns if any(token in str(c).lower() for token in pred_tokens)]
    conf_cols = [c for c in table.columns if any(token in str(c).lower() for token in confidence_tokens)]
    if actual_cols and pred_cols and len(table):
        actual_col = actual_cols[0]
        pred_col = next((c for c in pred_cols if c != actual_col), pred_cols[0])
        actual_num = pd.to_numeric(table[actual_col], errors='coerce')
        pred_num = pd.to_numeric(table[pred_col], errors='coerce')
        numeric_mask = actual_num.notna() & pred_num.notna()
        if numeric_mask.sum() >= 10:
            residual = actual_num[numeric_mask] - pred_num[numeric_mask]
            abs_error = residual.abs()
            print('
', path.name, '| MAE=', round(float(abs_error.mean()),5), '| RMSE=', round(float(np.sqrt(np.mean(residual**2))),5), '| bias=', round(float(residual.mean()),5))
            plt.figure(figsize=(7,5))
            plt.scatter(actual_num[numeric_mask], pred_num[numeric_mask], alpha=0.35, s=18)
            lo = min(actual_num[numeric_mask].min(), pred_num[numeric_mask].min())
            hi = max(actual_num[numeric_mask].max(), pred_num[numeric_mask].max())
            plt.plot([lo,hi],[lo,hi], linestyle='--')
            plt.xlabel(str(actual_col))
            plt.ylabel(str(pred_col))
            plt.title(f'Actual vs predicted — {path.name}')
            plt.tight_layout()
            plt.show()
            plt.figure(figsize=(7,4))
            plt.hist(residual, bins=30, alpha=0.82)
            plt.axvline(0, linestyle='--')
            plt.title(f'Residual distribution — {path.name}')
            plt.tight_layout()
            plt.show()
            worst_idx = abs_error.nlargest(min(15,len(abs_error))).index
            cols = list(dict.fromkeys([actual_col,pred_col]+conf_cols[:2]))
            worst = table.loc[worst_idx, cols].copy()
            worst['absolute_error'] = abs_error.loc[worst_idx].values
            display(worst.sort_values('absolute_error', ascending=False))
        else:
            agreement = table[actual_col].astype(str) == table[pred_col].astype(str)
            print('
', path.name, '| classification agreement=', round(float(agreement.mean()),4))
            if (~agreement).any():
                display(table.loc[~agreement, [actual_col,pred_col]+conf_cols[:2]].head(20))
    elif conf_cols:
        for col in conf_cols[:2]:
            values = pd.to_numeric(table[col], errors='coerce').dropna()
            if len(values) >= 10:
                plt.figure(figsize=(7,4))
                plt.hist(values, bins=30, alpha=0.82)
                plt.title(f'{col} distribution — {path.name}')
                plt.tight_layout()
                plt.show()


In [ ]:
# Display retained visual evidence from actual project runs
png_files = []
for base in [PROJECT/'results', PROJECT/'outputs', PROJECT/'artifacts', ROOT/'verified'/PROJECT_SLUG]:
    if base.exists():
        png_files.extend(sorted(base.rglob('*.png')))
print('Retained PNG figures:', len(png_files))
for path in png_files[:12]:
    try:
        image = plt.imread(path)
        plt.figure(figsize=(10,6))
        plt.imshow(image)
        plt.axis('off')
        plt.title(str(path.relative_to(ROOT)) if ROOT in path.parents else path.name)
        plt.tight_layout()
        plt.show()
    except Exception as exc:
        print('Could not display', path.name, '-', exc)


In [ ]:
# Reproducibility and evidence checklist
checks = [
    {'check':'README present', 'status':(PROJECT/'README.md').exists()},
    {'check':'Recruiter notebook present', 'status':(PROJECT/'project_notebook.ipynb').exists()},
    {'check':'Python implementation present', 'status':any(PROJECT.rglob('*.py'))},
    {'check':'Tests present', 'status':(PROJECT/'tests').exists() and any((PROJECT/'tests').rglob('test*.py'))},
    {'check':'Result/evidence files present', 'status':bool(candidate_files)},
    {'check':'Machine-readable JSON evidence', 'status':bool(json_files)},
    {'check':'Retained visual evidence', 'status':bool(png_files)},
]
checklist = pd.DataFrame(checks)
display(checklist)
print('Evidence checklist pass rate:', f"{100*checklist['status'].mean():.1f}%")
print('A failed item is a prompt to strengthen the project, not something to hide.')


# Engineering appendix — canonical application source

The analysis and visual evidence come first. The cells below preserve additional canonical Python from this project for reviewers who want to inspect pipelines, APIs, tests, feature code, monitoring and reusable implementation details.


## Canonical source: `src/diagnostics.py`


In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from typing import Iterable

import numpy as np
import pandas as pd
from scipy.stats import norm


@dataclass(frozen=True)
class EffectEstimate:
    segment: str
    treatment_rows: int
    control_rows: int
    estimate: float
    standard_error: float
    ci_low: float
    ci_high: float


@dataclass(frozen=True)
class BalanceCheck:
    variable: str
    treatment_mean: float
    control_mean: float
    standardized_difference: float
    passed: bool


def standardized_mean_difference(
    treatment_values: np.ndarray,
    control_values: np.ndarray,
) -> float:
    treatment_values = np.asarray(treatment_values, dtype=float)
    control_values = np.asarray(control_values, dtype=float)
    treatment_variance = float(np.var(treatment_values, ddof=1))
    control_variance = float(np.var(control_values, ddof=1))
    pooled_sd = float(np.sqrt((treatment_variance + control_variance) / 2.0))
    if pooled_sd == 0.0:
        return 0.0
    return float((treatment_values.mean() - control_values.mean()) / pooled_sd)


def numeric_balance_table(
    frame: pd.DataFrame,
    treatment_col: str,
    numeric_columns: Iterable[str],
    threshold: float = 0.10,
) -> pd.DataFrame:
    rows: list[dict[str, float | str | bool]] = []
    treatment_mask = frame[treatment_col].to_numpy() == 1
    for column in numeric_columns:
        values = frame[column].to_numpy(dtype=float)
        treated = values[treatment_mask]
        control = values[~treatment_mask]
        smd = standardized_mean_difference(treated, control)
        check = BalanceCheck(
            variable=column,
            treatment_mean=float(treated.mean()),
            control_mean=float(control.mean()),
            standardized_difference=smd,
            passed=bool(abs(smd) < threshold),
        )
        rows.append(asdict(check))
    return pd.DataFrame(rows)


def categorical_balance_table(
    frame: pd.DataFrame,
    treatment_col: str,
    categorical_column: str,
) -> pd.DataFrame:
    counts = (
        frame.groupby([treatment_col, categorical_column], observed=True)
        .size()
        .rename("rows")
        .reset_index()
    )
    totals = counts.groupby(treatment_col)["rows"].transform("sum")
    counts["share"] = counts["rows"] / totals
    pivot = counts.pivot(
        index=categorical_column,
        columns=treatment_col,
        values="share",
    ).fillna(0.0)
    for expected in (0, 1):
        if expected not in pivot.columns:
            pivot[expected] = 0.0
    output = pivot.rename(columns={0: "control_share", 1: "treatment_share"}).reset_index()
    output["absolute_share_gap"] = (
        output["treatment_share"] - output["control_share"]
    ).abs()
    return output.sort_values("absolute_share_gap", ascending=False)


def effect_estimate(
    outcome: np.ndarray,
    treatment: np.ndarray,
    segment: str,
    confidence: float = 0.95,
) -> EffectEstimate:
    outcome = np.asarray(outcome, dtype=float)
    treatment = np.asarray(treatment, dtype=int)
    treated = outcome[treatment == 1]
    control = outcome[treatment == 0]
    if len(treated) < 2 or len(control) < 2:
        raise ValueError("Both experiment arms need at least two observations")
    estimate = float(treated.mean() - control.mean())
    standard_error = float(
        np.sqrt(
            treated.var(ddof=1) / len(treated)
            + control.var(ddof=1) / len(control)
        )
    )
    alpha = 1.0 - confidence
    critical = float(norm.ppf(1.0 - alpha / 2.0))
    return EffectEstimate(
        segment=segment,
        treatment_rows=int(len(treated)),
        control_rows=int(len(control)),
        estimate=estimate,
        standard_error=standard_error,
        ci_low=float(estimate - critical * standard_error),
        ci_high=float(estimate + critical * standard_error),
    )


def segment_effect_table(
    frame: pd.DataFrame,
    segment_col: str = "segment",
    treatment_col: str = "treatment",
    outcome_col: str = "outcome",
) -> pd.DataFrame:
    rows: list[dict[str, float | str | int]] = []
    overall = effect_estimate(
        frame[outcome_col].to_numpy(),
        frame[treatment_col].to_numpy(),
        segment="overall",
    )
    rows.append(asdict(overall))
    for segment, group in frame.groupby(segment_col, observed=True):
        estimate = effect_estimate(
            group[outcome_col].to_numpy(),
            group[treatment_col].to_numpy(),
            segment=str(segment),
        )
        rows.append(asdict(estimate))
    return pd.DataFrame(rows)


def bootstrap_effect_distribution(
    outcome: np.ndarray,
    treatment: np.ndarray,
    rounds: int = 2000,
    seed: int = 42,
) -> np.ndarray:
    if rounds < 100:
        raise ValueError("Use at least 100 bootstrap rounds")
    rng = np.random.default_rng(seed)
    outcome = np.asarray(outcome, dtype=float)
    treatment = np.asarray(treatment, dtype=int)
    treated_index = np.where(treatment == 1)[0]
    control_index = np.where(treatment == 0)[0]
    draws = np.empty(rounds, dtype=float)
    for index in range(rounds):
        treated_sample = rng.choice(treated_index, size=len(treated_index), replace=True)
        control_sample = rng.choice(control_index, size=len(control_index), replace=True)
        draws[index] = float(
            outcome[treated_sample].mean() - outcome[control_sample].mean()
        )
    return draws


def randomization_inference_pvalue(
    outcome: np.ndarray,
    treatment: np.ndarray,
    permutations: int = 2000,
    seed: int = 42,
) -> float:
    if permutations < 100:
        raise ValueError("Use at least 100 permutations")
    rng = np.random.default_rng(seed)
    outcome = np.asarray(outcome, dtype=float)
    treatment = np.asarray(treatment, dtype=int)
    observed = abs(effect_estimate(outcome, treatment, "observed").estimate)
    more_extreme = 0
    for _ in range(permutations):
        shuffled = rng.permutation(treatment)
        candidate = abs(effect_estimate(outcome, shuffled, "permuted").estimate)
        if candidate >= observed:
            more_extreme += 1
    return float((more_extreme + 1) / (permutations + 1))


def bootstrap_summary(draws: np.ndarray) -> dict[str, float]:
    draws = np.asarray(draws, dtype=float)
    return {
        "mean": float(draws.mean()),
        "std": float(draws.std(ddof=1)),
        "ci_low_95": float(np.quantile(draws, 0.025)),
        "ci_high_95": float(np.quantile(draws, 0.975)),
        "probability_positive": float((draws > 0.0).mean()),
    }


def power_curve(
    outcome_sd: float,
    total_sample_sizes: Iterable[int],
    alpha: float = 0.05,
    target_effect: float = 2.5,
) -> pd.DataFrame:
    rows: list[dict[str, float | int]] = []
    critical = float(norm.ppf(1.0 - alpha / 2.0))
    for total_rows in total_sample_sizes:
        per_arm = max(int(total_rows) / 2.0, 2.0)
        standard_error = float(outcome_sd * np.sqrt(2.0 / per_arm))
        noncentrality = float(target_effect / standard_error)
        achieved_power = float(
            1.0
            - norm.cdf(critical - noncentrality)
            + norm.cdf(-critical - noncentrality)
        )
        minimum_detectable_effect = float(
            (critical + norm.ppf(0.80)) * standard_error
        )
        rows.append(
            {
                "total_rows": int(total_rows),
                "target_effect": float(target_effect),
                "achieved_power": achieved_power,
                "mde_at_80pct_power": minimum_detectable_effect,
            }
        )
    return pd.DataFrame(rows)


def guardrail_sensitivity_table(
    guardrail_estimate: float,
    guardrail_ci_low: float,
    effect_ci_low: float,
    margins: Iterable[float] = (-0.05, -0.10, -0.20, -0.30, -0.50),
) -> pd.DataFrame:
    rows: list[dict[str, float | str | bool]] = []
    for margin in margins:
        guardrail_pass = bool(guardrail_ci_low > margin)
        primary_pass = bool(effect_ci_low > 0.0)
        decision = "ship" if guardrail_pass and primary_pass else "hold"
        rows.append(
            {
                "non_inferiority_margin": float(margin),
                "guardrail_estimate": float(guardrail_estimate),
                "guardrail_ci_low": float(guardrail_ci_low),
                "primary_ci_low": float(effect_ci_low),
                "guardrail_pass": guardrail_pass,
                "primary_pass": primary_pass,
                "decision": decision,
            }
        )
    return pd.DataFrame(rows)


def build_diagnostics(
    frame: pd.DataFrame,
    adjusted_outcome: np.ndarray,
    seed: int = 42,
) -> dict[str, object]:
    numeric_balance = numeric_balance_table(
        frame,
        treatment_col="treatment",
        numeric_columns=["pre_metric"],
    )
    categorical_balance = categorical_balance_table(
        frame,
        treatment_col="treatment",
        categorical_column="segment",
    )
    segment_effects = segment_effect_table(frame)
    draws = bootstrap_effect_distribution(
        adjusted_outcome,
        frame["treatment"].to_numpy(),
        rounds=2000,
        seed=seed,
    )
    permutation_pvalue = randomization_inference_pvalue(
        adjusted_outcome,
        frame["treatment"].to_numpy(),
        permutations=1500,
        seed=seed + 1,
    )
    return {
        "numeric_balance": numeric_balance.to_dict(orient="records"),
        "categorical_balance": categorical_balance.to_dict(orient="records"),
        "segment_effects": segment_effects.to_dict(orient="records"),
        "bootstrap": bootstrap_summary(draws),
        "randomization_inference_pvalue": permutation_pvalue,
    }


# Portfolio depth check

**Meaningful code lines visible in this notebook:** 489. For a major recruiter-facing application the working target is roughly **1,000 meaningful lines**, with a practical guide of about 600–1,400 depending on the problem. This notebook is below the major-project guide and should grow only through substantive analysis/application depth.

Line count is not a quality metric by itself. Add code only when it improves the real project: data acquisition, validation, cleaning, EDA, visualisation, feature engineering, baselines, model comparison, tuning, leakage control, error analysis, explainability, uncertainty, inference, tests, monitoring, deployment or decision logic.
